About this notebook.

This notebook goes through all the textblocks and applies an additional layer of sanitization on them.

In [2]:
import spacy
import os
import glob
from spacy.tokens import Doc
from spacy.language import Language
import pickle
from unidecode import unidecode
import sddk
import pandas as pd
import re
import sys
import importlib
import json
from spacy.tokens import Token
from spacy.language import Language
import google_conf
import pandas as pd
import json
import fitz
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

In [3]:
source_path = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31/annotated_textblocks/"
len(os.listdir(source_path))

200

In [4]:
import shutil
shutil.copytree("/srv/data/tome/tome-corpus/EMLAP_2025-10-31/annotated_textblocks/", "../data/emlap_annotated_textblocks/", dirs_exist_ok=True)

'../data/emlap_annotated_textblocks/'

In [5]:
[f for f in sorted(os.listdir(source_path)) if "_params" not in f]

['100001_Augurello1515_Chrysopoeia_GB_Noscemus.json',
 '100002_Pseudo-Lull1518_De_secretis_naturae_MDZ_MBS.json',
 '100003_Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json',
 '100004_Anon1561_Verae_Alchemiae_MDZ_MBS.json',
 '100005_Pantheus1530_Voarchadumia_ONB.json',
 '100006_Savonarola1532_De_arte_conficiendi_aquam_vitae_ONB.json',
 '100007_Anon1550_Rosarium_philosophorum_ER_ZZ.json',
 '100008_Severinus1572_Epistola_MBZ_MBS.json',
 '100009_Vegius1518_Inter_inferiora_corpora_disputatio_ONB.json',
 '100010_Bracesco1548_De_alchemia_dialogi_duo_IA_Madrid.json',
 '100011_Anon1541_De_alchemia_MDZ_MBS.json',
 '100012_Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100014_Toxites1567_Spongia_stibii_MDZ_MBS.json',
 '100015_Gessner1569_Thesaurus_Euonymi_Philiatri_liber_secundus_MDZ_MBS.json',
 '100016_Bonus1546_Pretiosa_Margarita_Novella_ONB.json',
 '100017_Bodenstein1559_Isagoge_MDZ_MBS.json',
 '100018_Trevisanus1567_Pe

In [46]:
filename = '100092_Drebbel1621_Tractatus_duo_VD17_Halle.json'
filepath = os.path.join(source_path, filename)
with open(filepath, 'r', encoding='utf-8') as f:
    textblocks = json.load(f)

In [48]:
textblocks


[[],
 [],
 [],
 [],
 [{'coordinates': [125.27999877929688,
    158.63998413085938,
    354.7120056152344,
    163.4399871826172],
   'text': 'CORNELI DREBBELI,\n',
   'tag': 'text'},
  {'coordinates': [98.4000015258789,
    175.19998168945312,
    430.4165954589844,
    179.99998474121094],
   'text': 'CHEMICI & MECHANICI SVMMI\n',
   'tag': 'text'},
  {'coordinates': [132.72000122070312,
    200.42398071289062,
    347.3740234375,
    205.34397888183594],
   'text': 'TRACTATIS DUO:\n',
   'tag': 'title'},
  {'coordinates': [198.47999572753906,
    227.99996948242188,
    277.8348083496094,
    232.7999725341797],
   'text': 'PRIOR\n',
   'tag': 'text'},
  {'coordinates': [132.24000549316406,
    259.9440002441406,
    399.3508605957031,
    264.864013671875],
   'text': 'DE NATURA\n',
   'tag': 'title'},
  {'coordinates': [128.63999938964844,
    291.8399963378906,
    352.8864440917969,
    296.6399841308594],
   'text': 'ELEMENTORUM,\n',
   'tag': 'title'},
  {'coordinates': [93.360

In [8]:
def uniheader_textblocks(textblocks):
    """
    there is sometimes more than one automatically assigned header per page.
    If present, change the second header into normal text
    """
    textblocks_unheadered = []
    for p in textblocks:
        p_unheadered = []
        header_met = False
        for textblock in p:
            if textblock["tag"] == "header":
                if header_met:
                    textblock["tag"] = "text"
                else:
                    header_met = True
            p_unheadered.append(textblock)
        textblocks_unheadered.append(p_unheadered)
    return textblocks_unheadered

textblocks = uniheader_textblocks(textblocks)

In [9]:
textblocks[1][8:12]

[{'coordinates': [183.36000061035156,
   1033.4638671875,
   1669.21630859375,
   1038.3839111328125],
  'text': 'ARTIS LIBRO COMPREHENSARVM, AD¬\n',
  'tag': 'text'},
 {'coordinates': [240.9600067138672,
   1093.4400634765625,
   1615.7607421875,
   1098.239990234375],
  'text': 'IECTIS FORNACVM ET ALIORVM VASORVM FIGV¬\n',
  'tag': 'text'},
 {'coordinates': [349.9200134277344,
   1151.783935546875,
   1522.6253662109375,
   1156.7039794921875],
  'text': 'ris, partim ex impressis antehac autoribus, partim aliunde acce¬\n',
  'tag': 'text'},
 {'coordinates': [584.4000244140625,
   1199.9998779296875,
   1333.544921875,
   1204.7998046875],
  'text': 'ptis, & ex latibulis officinarum productis.\n',
  'tag': 'text'}]

In [83]:
import re
import unicodedata

SUPPORTED_LANG_TAGS = {"GR", "G", "F", "I", "H", "D"}
TAG_PATTERN = re.compile(r"\[(?P<open>[A-Za-z]+)]|\[/(?P<close>[A-Za-z]+)]")
S_PLACEHOLDER = "xyzxyzus"
BREAK_HYPHEN = "¬"

# ---------------------------------------------------------
# Latin-normalization helpers (used for S-content + main text)
# ---------------------------------------------------------
def latin_normalize_char(ch: str):
    """
    Normalize Latin text:
      - j/J → i/I
      - v/V → u/U
      - ſ → s
      - æ/Æ → ae/Ae
      - œ/Œ → oe/Oe
      - remove diacritics for Latin letters
      - keep Greek, symbols, numerals as-is
    """

    # Early-modern Latin OCR fixes
    if ch == "j":
        return "i"
    if ch == "J":
        return "I"
    if ch == "v":
        return "u"
    if ch == "V":
        return "U"

    # historical ligatures
    if ch == "ſ":
        return "s"
    if ch == "æ":
        return "ae"
    if ch == "Æ":
        return "Ae"
    if ch == "œ":
        return "oe"
    if ch == "Œ":
        return "Oe"

    # Only strip diacritics from Latin letters
    name = unicodedata.name(ch, "")
    if "LATIN" in name:
        decomposed = unicodedata.normalize("NFD", ch)
        base = "".join(c for c in decomposed if unicodedata.category(c) != "Mn")
        return unicodedata.normalize("NFC", base)

    return ch


def latin_normalize_str(s: str):
    # Character-wise normalization (diacritics, j/v, ligatures)
    s = "".join(latin_normalize_char(c) for c in s)
    # Multi-character corrections
    s = s.replace("ij", "ii")
    s = s.replace("IJ", "II")
    return s


# (kept for completeness, though we now filter via unicodedata)
ALLOWED_S_CHARS = re.compile(r"[^0-9A-Za-zα-ωΑ-Ω◉\s/+*\-]")


# =========================================================
# =                   SANITIZE TEXTBLOCK                  =
# =========================================================
def sanitize_textblock(tb):
    text = tb["text"]

    # ---------------------------------------------
    # 1. whitespace normalization
    # ---------------------------------------------
    text = text.replace("\xa0", " ")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    # ---------------------------------------------
    # 1B. Uppercase tag names
    # ---------------------------------------------
    def upper_tag(match):
        o = match.group("open")
        c = match.group("close")
        if o:
            return f"[{o.upper()}]"
        if c:
            return f"[/{c.upper()}]"
        return match.group(0)

    text = TAG_PATTERN.sub(upper_tag, text)

    # ---------------------------------------------
    # 1C. OCR double-bracket noise
    # ---------------------------------------------
    text = text.replace("[[", "[")
    text = text.replace("]]", "]")

    # ---------------------------------------------
    # 2. parse tags + repair structure
    # ---------------------------------------------
    out = []
    pos = 0
    lang_open = {L: 0 for L in SUPPORTED_LANG_TAGS}
    s_open = 0

    matches = list(TAG_PATTERN.finditer(text))

    for m in matches:
        start, end = m.span()
        out.append(text[pos:start])

        open_tag = m.group("open")
        close_tag = m.group("close")

        # S-tag open
        if open_tag == "S":
            s_open += 1
            out.append("[S]")

        # S-tag close
        elif close_tag == "S":
            if s_open == 0:
                out.append("[S]")  # create missing opener
            else:
                s_open -= 1
            out.append("[/S]")

        # language open
        elif open_tag in SUPPORTED_LANG_TAGS:
            lang_open[open_tag] += 1
            out.append(f"[{open_tag}]")

        # language close
        elif close_tag in SUPPORTED_LANG_TAGS:
            L = close_tag
            if lang_open[L] == 0:
                out.insert(0, f"[{L}]")  # missing opener
            else:
                lang_open[L] -= 1
            out.append(f"[/{L}]")

        pos = end

    out.append(text[pos:])
    sanitized = "".join(out)

    # ---------------------------------------------
    # 3. empty S spans
    # ---------------------------------------------
    sanitized = re.sub(r"\[S]\s*\[/S]", f"[S]{S_PLACEHOLDER}[/S]", sanitized)

    # ---------------------------------------------
    # 4. close unclosed language spans
    # ---------------------------------------------
    for L, cnt in lang_open.items():
        if cnt > 0:
            sanitized += f"[/{L}]" * cnt

    # ---------------------------------------------
    # 5. close unmatched S-openers
    # ---------------------------------------------
    sanitized = sanitized.replace("[S][/", "[S]xyzxyzus[/")
    while sanitized.count("[S]") > sanitized.count("[/S]"):
        sanitized += "[/S]"

    # ---------------------------------------------------------
    # 5B. PASS-1 over main text: just protect S-blocks (no normalization here)
    # ---------------------------------------------------------
    def protect_S_blocks(text):
        out = []
        i = 0
        in_S = False

        while i < len(text):
            if text.startswith("[S]", i):
                in_S = True
                out.append("[S]")
                i += 3
                continue

            if text.startswith("[/S]", i):
                in_S = False
                out.append("[/S]")
                i += 4
                continue

            out.append(text[i])
            i += 1

        return "".join(out)

    sanitized = protect_S_blocks(sanitized)

    # ---------------------------------------------
    # 6. CLEAN S-CONTENT
    # ---------------------------------------------
    def clean_s_inner(m):
        inner = m.group(1)
        # Normalize Latin + lowercase
        inner_norm = latin_normalize_str(inner.lower())

        out_chars = []
        for ch in inner_norm:
            cat = unicodedata.category(ch)
            if ch.isspace():
                out_chars.append(" ")
            elif cat.startswith("L"):   # letters
                out_chars.append(ch)
            elif cat.startswith("N"):   # digits
                out_chars.append(ch)
            elif cat.startswith("S"):   # symbols (keep ♣, etc.)
                out_chars.append(ch)
            else:
                # punctuation/control removed
                continue

        cleaned = "".join(out_chars)
        cleaned = re.sub(r"\s+", " ", cleaned).strip()
        # separate adjacent symbols: "♣♣" -> "♣ ♣"
        cleaned = re.sub(r"([^\w\s])([^\w\s])", r"\1 \2", cleaned)

        if cleaned == "":
            cleaned = S_PLACEHOLDER

        # exactly one leading and trailing space inside tags
        return f"[S] {cleaned} [/S]"

    sanitized = re.sub(r"\[S\](.*?)\[/S\]", clean_s_inner, sanitized)

    # ---------------------------------------------------------
    # X. MAIN-TEXT LATIN NORMALIZATION (ROMAN PROTECTED!)
    # ---------------------------------------------------------
    ROMAN_RE = re.compile(r"""
        ^M{0,4}
        (CM|CD|D?C{0,3})
        (XC|XL|L?X{0,3})
        (IX|IV|V?I{0,3})
        $
    """, re.VERBOSE)

    def normalize_main_text_segments(text):
        # split around tags, but keep tags as separate parts
        parts = re.split(r'(\[[A-Z/]+\])', text)

        out_parts = []
        inside_S = False

        for part in parts:

            if part.startswith("[S]"):
                inside_S = True
                out_parts.append(part)
                continue

            if part.startswith("[/S]"):
                inside_S = False
                out_parts.append(part)
                continue

            # other tags unchanged
            if re.fullmatch(r'\[[A-Z/]+\]', part):
                out_parts.append(part)
                continue

            # main text outside S: apply Latin normalization, protecting Roman numerals
            if not inside_S:
                raw_tokens = part.split(" ")
                new_tokens = []

                for tok in raw_tokens:
                    if ROMAN_RE.fullmatch(tok):
                        new_tokens.append(tok)
                    else:
                        new_tokens.append(latin_normalize_str(tok))

                out_parts.append(" ".join(new_tokens))
            else:
                # inside S, content already handled above
                out_parts.append(part)

        return "".join(out_parts)

    sanitized = normalize_main_text_segments(sanitized)

    # ---------------------------------------------
    # 7. spacing around tags (your proven logic)
    # ---------------------------------------------
    sanitized = sanitized.replace("[", " [")
    sanitized = sanitized.replace("]", "] ")
    sanitized = re.sub(r"\s+", " ", sanitized).strip()
    sanitized = re.sub(r"(\])(?! )", r"\1 ", sanitized)
    sanitized = re.sub(r"(?<! )(\[)", r" \1", sanitized)
    sanitized = re.sub(r"\s+", " ", sanitized)

    # ---------------------------------------------
    # 8. remove pre-punctuation space
    # ---------------------------------------------
    sanitized = re.sub(r"\s+([.,;:!?\)])", r"\1", sanitized)

    # ---------------------------------------------
    # 8B. uppercase-before-break-hyphen normalization
    # ---------------------------------------------
    def normalize_uppercase_continuation(text):
        out = []
        i = 0
        n = len(text)
        in_S = False

        while i < n:

            if text.startswith("[S]", i):
                in_S = True
                out.append("[S]")
                i += 3
                continue
            if text.startswith("[/S]", i):
                in_S = False
                out.append("[/S]")
                i += 4
                continue

            if not in_S:
                m = re.match(r"\b([A-Z]{2,})\b", text[i:])
                if m:
                    word = m.group(1)
                    end = i + len(word)

                    # normalize only when followed by BREAK_HYPHEN
                    if end < n and text[end] == BREAK_HYPHEN:
                        if not ROMAN_RE.match(word):
                            out.append(word.capitalize())
                        else:
                            out.append(word)
                    else:
                        out.append(word)

                    i += len(word)
                    continue

            out.append(text[i])
            i += 1

        return "".join(out)

    sanitized = normalize_uppercase_continuation(sanitized)

    # ---------------------------------------------
    # 9. block-boundary logic
    # ---------------------------------------------
    sanitized = sanitized.rstrip()

    if sanitized.endswith(BREAK_HYPHEN):
        sanitized = sanitized[:-1]
    else:
        sanitized += " "

    if len(sanitized) > 0 and sanitized[0] == " ":
        sanitized = sanitized[1:]

    # ---------------------------------------------
    # 10. return final
    # ---------------------------------------------
    out_tb = dict(tb)
    out_tb["text"] = sanitized
    return out_tb


def sanitize_all_textblocks(textblocks):
    return [[sanitize_textblock(tb) for tb in page] for page in textblocks]

In [84]:
def run_test(name, input_text, expected, sanitize_fn):
    tb = {"text": input_text, "tag": "text"}
    out = sanitize_fn(tb)["text"]

    if out == expected:
        print(f"PASS: {name}")
        print("  INPUT:    ", repr(input_text))
        print("  GOT:      ", repr(out))
    else:
        print(f"FAIL: {name}")
        print("  INPUT:    ", repr(input_text))
        print("  EXPECTED: ", repr(expected))
        print("  GOT:      ", repr(out))
    print("-" * 60)


tests = []


# 1 — Double bracket cleaning
tests.append((
    "double brackets",
    "[[S]]Aries[[/S]]",
    "[S] aries [/S] ",
))

# 2 — empty S → placeholder
tests.append((
    "empty S",
    "[S][/S]",
    "[S] xyzxyzus [/S] ",
))

# 3 — unmatched S opener
tests.append((
    "unmatched S opener",
    "[S]Mars",
    "[S] mars [/S] ",
))

# 4 — S cleaning
tests.append((
    "S cleaning",
    "[S] Mars!! ♣♣ [/S]",
    "[S] mars ♣ ♣ [/S] ",
))

# 5 — S latin normalize
tests.append((
    "S latin normalize",
    "[S]SÆCULŌrum ſumma[/S]",
    "[S] saeculorum summa [/S] ",
))

# 6 — uppercase normalize before BREAK_HYPHEN
tests.append((
    "upper normalize before BREAK_HYPHEN",
    "MAGISTER¬",
    "Magister",
))

# 7 — uppercase kept without BREAK_HYPHEN
tests.append((
    "uppercase kept without BREAK_HYPHEN",
    "MAGISTER",
    "MAGISTER ",
))

# 8 — roman numeral skip normalization
tests.append((
    "roman numeral skip normalization",
    "VIII",
    "VIII ",
))

# 9 — spacing around tags
tests.append((
    "tag spacing",
    "Aries[S]Mars[/S]Leo",
    "Aries [S] mars [/S] Leo ",
))

# 10 — language tags
tests.append((
    "language tags",
    "[gr]λόγος[/gr]",
    "[GR] λόγος [/GR] ",
))

# 11 — block boundary hyphen
tests.append((
    "block boundary hyphen",
    "finis¬",
    "finis",
))

# 12 — block boundary normal
tests.append((
    "block boundary normal",
    "finis",
    "finis ",
))

tests.append((
    "character replacements",
    " Jupitér venti",
    "Iupiter uenti ",
))

tests.append((
    "within tags character replacement",
    "Aries[S]Jupiter Venus[/S]Leo",
    "Aries [S] iupiter uenus [/S] Leo ",
))


for name, inp, exp in tests:
    run_test(name, inp, exp, sanitize_textblock)

PASS: double brackets
  INPUT:     '[[S]]Aries[[/S]]'
  GOT:       '[S] aries [/S] '
------------------------------------------------------------
PASS: empty S
  INPUT:     '[S][/S]'
  GOT:       '[S] xyzxyzus [/S] '
------------------------------------------------------------
PASS: unmatched S opener
  INPUT:     '[S]Mars'
  GOT:       '[S] mars [/S] '
------------------------------------------------------------
PASS: S cleaning
  INPUT:     '[S] Mars!! ♣♣ [/S]'
  GOT:       '[S] mars ♣ ♣ [/S] '
------------------------------------------------------------
PASS: S latin normalize
  INPUT:     '[S]SÆCULŌrum ſumma[/S]'
  GOT:       '[S] saeculorum summa [/S] '
------------------------------------------------------------
PASS: upper normalize before BREAK_HYPHEN
  INPUT:     'MAGISTER¬'
  GOT:       'Magister'
------------------------------------------------------------
PASS: uppercase kept without BREAK_HYPHEN
  INPUT:     'MAGISTER'
  GOT:       'MAGISTER '
-----------------------------

In [85]:
textblocks = sanitize_all_textblocks(textblocks)

In [86]:
textblocks[:6]

[[],
 [{'coordinates': [71.76000213623047,
    120.5040054321289,
    422.2099609375,
    125.42400360107422],
   'text': 'AUREOLI ',
   'tag': 'header'},
  {'coordinates': [106.31999969482422,
    160.79995727539062,
    380.797607421875,
    165.59996032714844],
   'text': 'THEOPHRASTI ',
   'tag': 'text'},
  {'coordinates': [151.67999267578125,
    188.63998413085938,
    330.93597412109375,
    193.4399871826172],
   'text': 'PARACELSI ',
   'tag': 'title'},
  {'coordinates': [68.63999938964844,
    239.51998901367188,
    418.5516357421875,
    244.3199920654297],
   'text': 'De summis Naturae mysteriis ',
   'tag': 'text'},
  {'coordinates': [135.60000610351562,
    270.4800109863281,
    352.5440673828125,
    275.2799987792969],
   'text': 'Commentarii tres, ',
   'tag': 'text'},
  {'coordinates': [224.16000366210938,
    315.6000061035156,
    257.32598876953125,
    320.3999938964844],
   'text': 'A ',
   'tag': 'title'},
  {'coordinates': [66.0,
    347.5199890136719,
    41

In [87]:
target_path = "../data/emlap_sanitized_textblocks/"
os.makedirs(target_path, exist_ok=True)

In [88]:
source_path = "../data/emlap_annotated_textblocks/"
len(os.listdir(source_path)) # 100 for textblocks, 100 for parameters used for their extraction

200

In [89]:
os.listdir(source_path)

['100083_Sendivogius1616_Tractatus_de_sulphure_MBS_MDZ_params.json',
 '100084_Croll1609_Basilica_chymica_MDZ_MBS.json',
 '100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json',
 '100099_Francus1607_De_Arte_Chemica_VD17_Halle_params.json',
 '100044_Dorn1578_Theophrasti_Germani_Principis_MDZ_MBS_params.json',
 '100058_Hagecius1596_Actio_medica_ER_UBB.json',
 '100078_Harvet1605_Demonstratio_veritatis_VD17_FAU_params.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100069_Severinus1571_Idea_medicinae_philosophicae_GB_Noscemus.json',
 '100074_Hoghelande1595_De_lapidis_physici_conditionibus_MDZ_MBS.json',
 '100070_Libavius1597_Alchemia_ER_Noscemus_params.json',
 '100066_Suchten1575_De_secretis_antimonii_ONB_params.json',
 '100059_Suavius1567_Theophrasti_Paracelsi_Philosophiae_ONB.json',
 '100053_Mirandola1586_De_auro_libri_tres_MDZ_MBS.json',
 '100072_Andernach1571_De_medicina_veteri_et_novi_MDZ_MBS_params.json',
 '100080_Anon1611_Tratatus_de_secretissimo_MDZ_MBS_params.jso

In [90]:
for filename in os.listdir(source_path):
    if "_params" not in filename:
            try:
                filepath = os.path.join(source_path, filename)
                with open(filepath, 'r', encoding='utf-8') as f:
                    textblocks = json.load(f)
                textblocks = uniheader_textblocks(textblocks)
                textblocks = sanitize_all_textblocks(textblocks)
                with open(target_path + filename, "w") as f:
                    json.dump(textblocks, f)
            except:
                print("failed with file: ", filename)
                pass